# Manufacturing Data Analysis EDA Skeleton

## Project Context

This notebook is a minimal starting point for later exploratory data analysis and project presentation work.

Current goal:

- understand equipment, line, shift, quality, and failure behavior from the current processed manufacturing dataset
- keep the analysis aligned with the current repository structure and contract tests

Important boundary:

- several KPI fields in this project, including `planned_production`, `actual_production`, `defect_rate`, `availability`, `performance`, `quality_rate`, and `oee`, are current project proxy / simulated metrics
- this notebook should support structured analysis, but not claim industrial-grade plant conclusions at this stage


## Analysis Questions

This notebook is designed to help answer questions like:

1. Which equipment or production lines show weaker OEE performance?
2. Do `Day`, `Evening`, and `Night` shifts show different KPI patterns?
3. How does machine failure relate to quality and OEE behavior?
4. Which equipment appears more often in failure-labelled records?
5. Which insights are supported by current proxy metrics, and which conclusions are still outside the scope of this project?


In [ ]:
from pathlib import Path

import pandas as pd

## Data Loading

Choose one processed CSV source.

- `manufacturing_data_processed.csv` is the current legacy reference
- `data/processed/manufacturing_data_processed_refactor.csv` is the current refactor snapshot

The default below uses the legacy processed CSV.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

legacy_processed_path = PROJECT_ROOT / 'manufacturing_data_processed.csv'
refactor_processed_path = PROJECT_ROOT / 'data' / 'processed' / 'manufacturing_data_processed_refactor.csv'

# Change this path if you want to inspect the refactor snapshot instead.
data_path = legacy_processed_path

print(f'Using dataset: {data_path}')
df = pd.read_csv(data_path)

print(f'Shape: {df.shape}')
display(df.head())

In [ ]:
df.columns.tolist()

## Dataset Overview

Start with the most basic checks before writing any interpretation.

Suggested questions:

- How many rows and columns are there?
- How many unique equipment IDs and production lines are there?
- What is the current date range?
- Are the key KPI columns present and non-empty?


In [ ]:
overview = {
    'row_count': len(df),
    'column_count': len(df.columns),
    'equipment_count': df['equipment_id'].nunique(),
    'production_line_count': df['production_line'].nunique(),
    'shift_values': sorted(df['shift'].dropna().unique().tolist()),
    'min_production_time': df['production_time'].min(),
    'max_production_time': df['production_time'].max(),
}

overview

## Equipment / Line KPI Analysis

Use this section to compare equipment and production lines.

Suggested focus:

- mean `oee`
- mean `availability`
- mean `performance`
- mean `quality_rate`
- total `defect_count`


In [ ]:
equipment_kpi = (
    df.groupby(['production_line', 'equipment_id'], as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_defect_count=('defect_count', 'sum'),
    )
    .sort_values(['avg_oee', 'total_defect_count'], ascending=[True, False])
)

equipment_kpi.head(10)

## Shift KPI Analysis

Use this section to compare `Day`, `Evening`, and `Night` shifts.

Suggested focus:

- average `oee`
- average `availability`
- average `performance`
- average `quality_rate`
- total `defect_count`
- failure-labelled record count


In [ ]:
shift_kpi = (
    df.groupby(['production_line', 'shift'], as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_defect_count=('defect_count', 'sum'),
        failure_records=('Machine failure', 'sum'),
    )
    .sort_values(['production_line', 'shift'])
)

shift_kpi

## Failure Summary

Use this section to inspect failure-labelled records and compare them with non-failure records.

Suggested focus:

- number of failure-labelled records
- average `oee` under failure vs non-failure records
- average `quality_rate` under failure vs non-failure records
- equipment with more failure-labelled records


In [ ]:
failure_summary = df.groupby('Machine failure', as_index=False).agg(
    record_count=('UDI', 'count'),
    avg_oee=('oee', 'mean'),
    avg_quality_rate=('quality_rate', 'mean'),
    avg_defect_rate=('defect_rate', 'mean'),
)

failure_by_equipment = (
    df.groupby('equipment_id', as_index=False)
    .agg(
        failure_records=('Machine failure', 'sum'),
        avg_oee=('oee', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
    )
    .sort_values(['failure_records', 'avg_oee'], ascending=[False, True])
)

display(failure_summary)
display(failure_by_equipment.head(10))

## Findings Draft

Use this section to draft short findings in plain language.

Finding 1

- Observation:
- Evidence:
- Business meaning:
- Limitation:

Finding 2

- Observation:
- Evidence:
- Business meaning:
- Limitation:

Finding 3

- Observation:
- Evidence:
- Business meaning:
- Limitation:


## Limitations

Keep these limits visible when writing conclusions:

- current OEE, quality, and production fields are proxy / simulated metrics under current project rules
- current timestamps, lines, and equipment identifiers are partly generated by the preprocessing logic
- this notebook supports structured local analysis, not industrial-grade operational claims
- later notebook and dashboard work should stay aligned with `docs/analysis_report_template.md`
